# V2 Phase 13 — Colab GPU DEV calibration (lock T)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Runs the frozen FinQA **dev** 40-question calibration set through **multi_agent_uq** only, then locks T with the pre-registered rule:

- maximise selective accuracy
- subject to coverage ≥ 0.50
- tie-break: lowest T

Uses **llama_cpp + Qwen3-8B**. Does **not** use the frozen 140. Does **not** run the 420-case benchmark. Does **not** start Phase 14+.

## Setup

Push latest V2 (Phase 13) to branch `cursor/empty-v2-workspace`, then run **this notebook on Colab GPU**.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

**Outputs:** `results/raw/phase13_calibration/{run_id}/cases.jsonl`, `results/config/threshold.lock.json`

If Colab disconnects: `--resume-latest`. Do not restart from question 1.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError('Open this notebook on Colab GPU. Do not run Phase 13 calibration on the Mac.')

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'run_calibration.py').is_file():
    raise FileNotFoundError(f'Phase 13 script missing at {V2_ROOT}. Push Phase 13 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 13 calibration (`llama_cpp`, 40 DEV UQ cases)

If this cell was interrupted, use section 5b instead.

In [ ]:
import os
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
!PYTHONPATH=. python scripts/run_calibration.py --backend llama_cpp --n-questions 40

## 5b. Resume after disconnect

In [ ]:
# Uncomment only after an interrupted run:
# import os
# os.environ['V2_REQUIRE_CUDA'] = '1'
# !PYTHONPATH=. python scripts/run_calibration.py --backend llama_cpp --n-questions 40 --resume-latest --retry-failed

## 6. Confirm lock file (DEV only, not the frozen 140)

In [ ]:
import json
from pathlib import Path

summary = Path('results/config/phase13_calibration_summary.json')
lock = Path('results/config/threshold.lock.json')
cand = Path('results/config/threshold.candidate.json')
print('summary', summary.is_file(), 'lock', lock.is_file(), 'candidate', cand.is_file())
data = json.loads(summary.read_text())
print('status', data.get('status'), 'backend', data.get('backend'), 'device', data.get('device'))
print('n_completed', data.get('n_completed'), 'lock', data.get('lock'))
if data.get('device') == 'mps_capable_host':
    raise RuntimeError('Mac result, not Colab T4.')
if data.get('used_frozen_test_140') is True:
    raise RuntimeError('Calibration must not use the frozen 140.')
ids = data.get('question_ids') or []
if any(str(i).startswith('finqa_test_') for i in ids):
    raise RuntimeError('Test IDs in calibration run.')
if not lock.is_file():
    raise RuntimeError('Official threshold.lock.json was not written. Check CUDA/llama_cpp and n=40.')
lock_data = json.loads(lock.read_text())
print('LOCKED T', lock_data.get('threshold'), 'coverage', lock_data.get('coverage'), 'sel_acc', lock_data.get('selective_accuracy'))
if lock_data.get('locked') is not True:
    raise RuntimeError('threshold.lock.json exists but locked is not True.')
if lock_data.get('used_frozen_test_140') is True:
    raise RuntimeError('Lock file reports test leakage.')
print('Phase 13 official lock OK')

## 7. Save Phase 13 results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
DRIVE = Path('/content/drive/MyDrive/MSc-RAG')

summary = json.loads((V2 / 'results' / 'config' / 'phase13_calibration_summary.json').read_text())
run_id = summary['run_id']

raw_src = V2 / 'results' / 'raw' / 'phase13_calibration' / run_id
raw_dest = DRIVE / 'results' / 'raw' / 'phase13_calibration' / run_id
raw_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(raw_src, raw_dest, dirs_exist_ok=True)
print('copied raw', raw_dest)

cfg_dest = DRIVE / 'configs' / 'phase13'
cfg_dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase13_runtime_fingerprint.json',
    'phase13_smoke_test.json',
    'phase13_calibration_summary.json',
    'threshold.lock.json',
    'threshold.candidate.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, cfg_dest / name)
        print('copied', name)